# House Price Prediction with TensorFlow

## Simple Linear Regression for Beginners

This project predicts a house price using one input feature:

- **Input:** House Size in square feet
- **Output:** House Price

Because there is one input and one output, we will use simple linear regression.

## Regression Evaluation

Regression predicts a number, so we do not use classification accuracy.

- **MAE:** Average prediction error in price
- **R² Score:** How well the model explains the relationship

An R² score closer to 100% means the predictions follow the data closely.

## 1. Install TensorFlow if Required

Run the next command only when TensorFlow is not installed. Remove the first # symbol before running it.

In [ ]:
# !pip install tensorflow

## 2. Import the Libraries

Pandas loads the dataset, NumPy manages numerical values, Matplotlib creates charts and TensorFlow builds the model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

## 3. Set the Random Seed

The random seed helps produce similar results each time the notebook runs.

In [ ]:
tf.keras.utils.set_random_seed(42)

## 4. Load the Dataset

This path loads the CSV file created by the separate dataset-generation code.

In [ ]:
file_path = r"C:\Users\Muhammad Huzifa\Downloads\house_price_dataset.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")

## 5. View the First Rows

The dataset contains only Size and Price.

In [ ]:
df.head()

## 6. Check the Dataset Size

The dataset should contain 500 rows and 2 columns.

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

## 7. Check the Data Types

Both columns should contain numerical values.

In [ ]:
df.info()

## 8. Check Missing Values

A clean generated dataset should have zero missing values.

In [ ]:
df.isnull().sum()

## 9. View the Statistical Summary

Describe shows the minimum, maximum, average and other statistics.

In [ ]:
df.describe()

## 10. Visualize Size and Price

The scatter plot should show that larger houses usually have higher prices.

In [ ]:
plt.scatter(df["Size"], df["Price"], color="#2A9D8F", alpha=0.7)

plt.title("House Size and Price")
plt.xlabel("House Size in Square Feet")
plt.ylabel("House Price")
plt.show()

## 11. Separate the Input and Output

X contains the house sizes. y contains the prices.

Price is divided by 1,000 during training to keep the numbers smaller. It will be converted back later.

In [ ]:
X = df[["Size"]].values.astype("float32")
y = (df["Price"].values / 1000).astype("float32")

print("Input shape:", X.shape)
print("Output shape:", y.shape)

## 12. Split the Dataset

- **80% training data:** Used to learn the relationship
- **20% testing data:** Used to evaluate unseen predictions

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## 13. Normalize the House Sizes

Normalization places the input values on a smaller scale and helps the model train smoothly.

In [ ]:
size_normalizer = tf.keras.layers.Normalization(axis=None)

size_normalizer.adapt(X_train)

## 14. Build the Linear Regression Model

The model contains one Dense neuron because it predicts one price.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(1,)),
    size_normalizer,
    tf.keras.layers.Dense(1)
])

## 15. Compile the Model

- **SGD:** Updates the model parameters
- **MSE:** Measures squared prediction errors
- **MAE:** Measures the average prediction error

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
    loss="mse",
    metrics=["mae"]
)

## 16. View the Model Structure

The summary shows the input, normalization layer and one output neuron.

In [ ]:
model.summary()

## 17. Train the Model

One epoch means that the model has studied the training data once.

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=150,
    batch_size=32,
    validation_split=0.20,
    verbose=0
)

print("Model training completed.")

## 18. Plot the Training Loss

A decreasing loss shows that the model is learning.

In [ ]:
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("Model Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.legend()
plt.show()

## 19. Evaluate the Model

MAE tells us the average difference between the actual and predicted prices.

In [ ]:
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)

mae_in_price = test_mae * 1000

print("Test MSE:", test_loss)
print("Average price error:", round(mae_in_price, 2))

## 20. Predict the Test Prices

The model predicts prices for houses it did not use during training.

In [ ]:
predicted_prices = model.predict(X_test, verbose=0).flatten() * 1000
actual_prices = y_test * 1000

## 21. Compare Actual and Predicted Prices

The table shows the real price and the model's prediction.

In [ ]:
results = pd.DataFrame({
    "Size": X_test.flatten(),
    "Actual Price": actual_prices,
    "Predicted Price": predicted_prices
})

results = results.round(2)

results.head(10)

## 22. Calculate the R² Score

R² shows how closely the predictions follow the actual house prices.

In [ ]:
r2 = r2_score(actual_prices, predicted_prices)

print("R² Score:", round(r2, 4))
print("R² Percentage:", round(r2 * 100, 2), "%")

## 23. Plot Actual Prices Against Predicted Prices

Predictions close to the diagonal line are accurate.

In [ ]:
lowest_price = min(actual_prices.min(), predicted_prices.min())
highest_price = max(actual_prices.max(), predicted_prices.max())

plt.scatter(actual_prices, predicted_prices, color="#E76F51", alpha=0.7)
plt.plot(
    [lowest_price, highest_price],
    [lowest_price, highest_price],
    color="black",
    linestyle="--"
)

plt.title("Actual Prices vs Predicted Prices")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.show()

## 24. Display the Regression Line

The line shows the price trend learned by the model.

In [ ]:
size_line = np.linspace(df["Size"].min(), df["Size"].max(), 100).reshape(-1, 1)
price_line = model.predict(size_line, verbose=0).flatten() * 1000

plt.scatter(df["Size"], df["Price"], color="#A8DADC", alpha=0.6, label="Dataset")
plt.plot(size_line, price_line, color="#E63946", linewidth=3, label="Regression Line")

plt.title("House Price Regression Line")
plt.xlabel("House Size in Square Feet")
plt.ylabel("House Price")
plt.legend()
plt.show()

## 25. Predict the Price of a New House

This example predicts the price of a 2,000 square-foot house.

In [ ]:
new_house_size = np.array([[2000]], dtype="float32")

new_price = model.predict(new_house_size, verbose=0)[0][0] * 1000

print("House size:", new_house_size[0][0], "square feet")
print("Predicted price:", round(new_price, 2))

## 26. Save the Trained Model

The trained model is saved in the Downloads folder.

In [ ]:
model_path = r"C:\Users\Muhammad Huzifa\Downloads\house_price_linear_regression.keras"

model.save(model_path)

print("Model saved successfully.")

## 27. Student Practice

1. Predict the price of a 1,500 square-foot house.
2. Predict the price of a 3,000 square-foot house.
3. Change the number of epochs from 150 to 100.
4. Compare the new MAE and R² score.
5. Change the training and testing split to 70% and 30%.

## Notebook Completed

You have loaded a house-price dataset, trained a TensorFlow linear-regression model, evaluated its predictions and predicted the price of a new house.